# RES-142 CPU-only Google Drive storage qualification (Colab 2026.04)

**Preparation artifact only. Every code cell is unexecuted. No current authorization to connect Colab or mount Drive is implied.**

Before running any cell, manually select Colab runtime **2026.04** and **CPU / no accelerator** in Runtime settings. The notebook cannot attest those UI selections. Do not change the runtime to GPU.

Before accepting the Drive mount prompt, visually verify the intended Google account; use that same account for the independent Drive UI check.

This is a separate, minimal Drive-storage probe, not the G4-only B1 notebook. It installs no packages, loads no model, uses no GPU, transfers no dataset, and runs no benchmark. It asks for two independent authorization phrases before mounting Drive. The only Drive location it creates or reads is `/content/drive/MyDrive/DynamisRAG/RES-142/drive-qualifications/<unique_run_id>/`.

After mount, it writes one synthetic sentinel and one manifest with the reviewed stdlib-only support helper, verifies read-back SHA-256, and calls Drive flush/unmount. It then requires an independent Drive UI check: download exactly the two files from the printed unique folder and upload those copies here for a second SHA-256 comparison. Qualification reports `complete` only when writes, flush/unmount, and both independent UI hashes succeed. Any failed or incomplete step returns `incomplete` and stops.

The first code cell asks you to upload the exact `notebooks/res142_b1_support.py` from this reviewed branch and verifies its canonical Git-LF byte count and SHA-256 against the preflight documentation. It does not mount or inspect Drive.

## 1. Upload and verify the reviewed support helper

This cell reads one local helper upload only. It does not connect to or mount Drive.

In [ ]:
from __future__ import annotations

import hashlib
import json
import sys
import types

from google.colab import files

EXPECTED_SUPPORT_BYTES = 34065
EXPECTED_SUPPORT_SHA256 = "264ff767ee3f6d22fa295a48281201b3f52834f55f958b688758e09d8f569200"

uploaded_support = files.upload()
if set(uploaded_support) != {"res142_b1_support.py"}:
    raise RuntimeError("Upload only the reviewed res142_b1_support.py file.")
support_bytes = uploaded_support["res142_b1_support.py"]
observed_support_sha256 = hashlib.sha256(support_bytes).hexdigest()
if len(support_bytes) != EXPECTED_SUPPORT_BYTES or observed_support_sha256 != EXPECTED_SUPPORT_SHA256:
    raise RuntimeError("Support helper byte count or SHA-256 differs from the reviewed artifact.")

support_module = types.ModuleType("res142_b1_support")
support_module.__file__ = "res142_b1_support.py"
exec(compile(support_bytes, support_module.__file__, "exec"), support_module.__dict__)
sys.modules[support_module.__name__] = support_module
support = support_module
print(json.dumps({"support_helper_status": "verified", "size_bytes": len(support_bytes), "sha256": observed_support_sha256}, sort_keys=True))

## 2. Define the fail-closed qualification flow

The authorization and exact destination are validated before `drive.mount()`. The helper writes only into a new directory under the reviewed root. No Drive directory listing is performed.

In [ ]:
import json
import time
from collections.abc import Callable, Mapping
from pathlib import Path

CPU_QUALIFICATION_AUTHORIZATION_REQUIRED = "APPROVE RES-142 CPU-ONLY DRIVE QUALIFICATION"
QUALIFICATION_DIRECTORY_NAME = "drive-qualifications"
SENTINEL_FILENAME = "sentinel.json"
MANIFEST_FILENAME = "qualification-manifest.json"
DRIVE_FLUSH_TIMEOUT_MS = 30000


def run_drive_storage_qualification(
    *,
    runtime_confirmation: str,
    accelerator_confirmation: str,
    qualification_authorization: str,
    drive_authorization: str,
    destination_confirmation: str,
    mountpoint: Path,
    drive_root: Path,
    local_staging_root: Path,
    drive_mount: Callable[[str], object],
    drive_is_mounted: Callable[[Path], bool],
    drive_flush_and_unmount: Callable[..., object],
    verify_in_drive_ui: Callable[[str, Mapping[str, str]], Mapping[str, object]],
) -> dict[str, object]:
    if runtime_confirmation != support.COLAB_RUNTIME_VERSION_REQUIRED:
        raise PermissionError("Select and confirm Colab runtime 2026.04 before mounting Drive.")
    if accelerator_confirmation != "CPU":
        raise PermissionError("Select and confirm CPU / no accelerator before mounting Drive.")
    if qualification_authorization != CPU_QUALIFICATION_AUTHORIZATION_REQUIRED:
        raise PermissionError("CPU Drive qualification requires its separate explicit authorization.")
    support.validate_drive_storage_authorization(
        authorization=drive_authorization,
        destination_confirmation=destination_confirmation,
    )
    if mountpoint != support.COLAB_DRIVE_MOUNTPOINT or drive_root != support.COLAB_DRIVE_ROOT:
        raise ValueError("CPU qualification must use the reviewed RES-142 Drive root.")

    run_id = support.new_drive_run_id()
    qualification_root = drive_root / QUALIFICATION_DIRECTORY_NAME
    run_dir = qualification_root / run_id
    local_staging_dir = local_staging_root / run_id
    state: dict[str, object] = {
        "backend": "google_drive",
        "run_id": run_id,
        "mountpoint": str(mountpoint),
        "required_root": str(drive_root),
        "run_dir": str(run_dir),
        "local_staging_dir": str(local_staging_dir),
        "persistence_status": "preflight",
        "persisted_artifacts": {},
        "errors": [],
    }
    errors: list[dict[str, str]] = []
    report: dict[str, object] = {
        "scope": "cpu-only-drive-storage",
        "status": "incomplete",
        "qualification_id": run_id,
        "runtime_version_required": support.COLAB_RUNTIME_VERSION_REQUIRED,
        "drive_root": str(drive_root),
        "run_directory": str(run_dir),
        "write_readback_status": "not-run",
        "drive_flush_status": "not-attempted",
        "drive_ui_verification_status": "not-run",
        "files": {},
        "errors": errors,
    }
    mount_attempted = False
    mount_verified = False
    stage = "drive_mount"
    sentinel_record: dict[str, object] | None = None
    manifest_record: dict[str, object] | None = None

    try:
        mount_attempted = True
        drive_mount(str(mountpoint))
        if not drive_is_mounted(mountpoint):
            raise RuntimeError(f"Google Drive is not mounted at {mountpoint}.")
        mount_verified = True

        my_drive = mountpoint / "MyDrive"
        dynamis_root = my_drive / "DynamisRAG"
        if not my_drive.is_dir():
            raise RuntimeError(f"Mounted Drive has no MyDrive directory at {my_drive}.")
        if my_drive.is_symlink() or dynamis_root.is_symlink() or drive_root.is_symlink():
            raise ValueError("Drive qualification path must not contain symlinked components.")
        if drive_root != dynamis_root / "RES-142":
            raise ValueError("Drive qualification path differs from the reviewed RES-142 root.")
        if not drive_root.exists():
            drive_root.mkdir(parents=True, exist_ok=True)
        if not drive_root.is_dir() or not drive_root.resolve().is_relative_to(my_drive.resolve()):
            raise ValueError("Reviewed Drive root is not a directory inside MyDrive.")
        if qualification_root.is_symlink():
            raise ValueError("Drive qualification directory must not be a symlink.")
        qualification_root.mkdir(exist_ok=True)
        if not qualification_root.resolve().is_relative_to(drive_root.resolve()):
            raise ValueError("Drive qualification directory resolves outside the reviewed root.")
        run_dir.mkdir(parents=False, exist_ok=False)
        if run_dir.is_symlink() or not run_dir.resolve().is_relative_to(qualification_root.resolve()):
            raise ValueError("Unique qualification directory resolves outside its parent.")

        sentinel_data = (
            json.dumps(
                {
                    "schema": "res142-cpu-drive-sentinel-v1",
                    "qualification_id": run_id,
                    "kind": "synthetic-drive-storage-sentinel",
                    "created_at_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
                },
                sort_keys=True,
                separators=(",", ":"),
            ).encode("utf-8")
            + b"\n"
        )
        stage = "sentinel_write_readback"
        sentinel_record = support.persist_drive_bytes(
            state, local_staging_dir, SENTINEL_FILENAME, sentinel_data
        )
        if sentinel_record.get("status") != "verified" or sentinel_record.get("sha256") != sentinel_record.get("readback_sha256"):
            raise OSError("Sentinel Drive read-back did not verify.")
        report["write_readback_status"] = "sentinel-verified"
        report["files"] = {SENTINEL_FILENAME: sentinel_record}

        manifest = {
            "schema": "res142-cpu-drive-qualification-v1",
            "qualification_id": run_id,
            "scope": "cpu-only-drive-storage",
            "runtime_version_required": support.COLAB_RUNTIME_VERSION_REQUIRED,
            "runtime_version_verification": "operator_confirms_2026.04_in_colab_ui",
            "runtime_version_confirmation": runtime_confirmation,
            "accelerator_confirmation": accelerator_confirmation,
            "drive_root": str(drive_root),
            "run_directory": str(run_dir),
            "write_readback_status": "verified",
            "drive_flush_status": "pending",
            "drive_ui_verification_status": "pending",
            "sentinel": {
                "path": sentinel_record["path"],
                "size_bytes": sentinel_record["size_bytes"],
                "sha256": sentinel_record["sha256"],
                "readback_sha256": sentinel_record["readback_sha256"],
            },
        }
        stage = "manifest_write_readback"
        manifest_record = support.persist_drive_json(
            state, local_staging_dir, MANIFEST_FILENAME, manifest
        )
        if manifest_record.get("status") != "verified" or manifest_record.get("sha256") != manifest_record.get("readback_sha256"):
            raise OSError("Qualification manifest Drive read-back did not verify.")
        report["write_readback_status"] = "verified"
        report["files"][MANIFEST_FILENAME] = manifest_record
    except Exception as exc:
        state["persistence_status"] = "incomplete"
        errors.append({"stage": stage, "error": support.sanitize_output(exc, limit=1000)})
    finally:
        if mount_attempted:
            try:
                drive_flush_and_unmount(timeout_ms=DRIVE_FLUSH_TIMEOUT_MS)
                state["drive_flush_status"] = "flushed"
                report["drive_flush_status"] = "flushed"
            except Exception as exc:
                state["drive_flush_status"] = "incomplete"
                state["persistence_status"] = "incomplete"
                report["drive_flush_status"] = "incomplete"
                errors.append(
                    {"stage": "drive_flush_unmount", "error": support.sanitize_output(exc, limit=1000)}
                )

    if not errors and mount_verified and sentinel_record is not None and manifest_record is not None:
        expected_hashes = {
            SENTINEL_FILENAME: str(sentinel_record["sha256"]),
            MANIFEST_FILENAME: str(manifest_record["sha256"]),
        }
        try:
            if report["drive_flush_status"] != "flushed":
                raise RuntimeError("Drive flush/unmount did not complete.")
            ui_result = verify_in_drive_ui(str(run_dir), expected_hashes)
            if ui_result.get("files_visible") is not True:
                raise RuntimeError("Independent Drive UI verification did not confirm both files.")
            for filename, expected_hash in expected_hashes.items():
                if ui_result.get(f"{filename}_sha256") != expected_hash:
                    raise OSError(f"Independent Drive UI SHA-256 mismatch for {filename}.")
            report["drive_ui_verification_status"] = "verified"
        except Exception as exc:
            state["persistence_status"] = "incomplete"
            errors.append(
                {"stage": "independent_drive_ui_verification", "error": support.sanitize_output(exc, limit=1000)}
            )

    complete = (
        not errors
        and report["write_readback_status"] == "verified"
        and report["drive_flush_status"] == "flushed"
        and report["drive_ui_verification_status"] == "verified"
    )
    state["persistence_status"] = "complete" if complete else "incomplete"
    report["persistence_status"] = state["persistence_status"]
    report["status"] = "complete" if complete else "incomplete"
    return report

## 3. Explicit authorization, probe, and independent UI verification

The probe flushes and unmounts before the separate Drive UI check. The UI check downloads only the two named files and compares both SHA-256 values.

In [ ]:
import hashlib
import json
from pathlib import Path

from google.colab import drive as colab_drive, files


def verify_in_independent_drive_ui(
    run_directory: str, expected_hashes: dict[str, str]
) -> dict[str, object]:
    print(
        "In a separate Google Drive UI tab for the same account, open this exact folder and "
        f"download only the two named files: {run_directory}"
    )
    ui_confirmation = input(
        "After independently confirming both files are visible in Drive UI, type DRIVE UI VERIFIED: "
    ).strip()
    if ui_confirmation != "DRIVE UI VERIFIED":
        raise RuntimeError("Independent Drive UI visibility confirmation is required.")
    uploaded_files = files.upload()
    if set(uploaded_files) != set(expected_hashes):
        raise RuntimeError("Upload exactly sentinel.json and qualification-manifest.json from Drive UI.")
    observed_hashes = {
        filename: hashlib.sha256(file_bytes).hexdigest()
        for filename, file_bytes in uploaded_files.items()
    }
    return {
        "files_visible": True,
        **{f"{filename}_sha256": digest for filename, digest in observed_hashes.items()},
    }


runtime_confirmation = input(
    "After visually confirming runtime 2026.04 in Colab, type 2026.04: "
).strip()
accelerator_confirmation = input(
    "After visually confirming Hardware accelerator None, type CPU: "
).strip()
qualification_authorization = input(
    "Type APPROVE RES-142 CPU-ONLY DRIVE QUALIFICATION to authorize the CPU-only probe: "
).strip()
drive_authorization = input(
    f"Type {support.COLAB_DRIVE_AUTHORIZATION_REQUIRED} to authorize Drive storage: "
).strip()
destination_confirmation = input(
    f"Type the exact reviewed Drive root {support.COLAB_DRIVE_ROOT.as_posix()}: "
).strip()

try:
    qualification_result = run_drive_storage_qualification(
        runtime_confirmation=runtime_confirmation,
        accelerator_confirmation=accelerator_confirmation,
        qualification_authorization=qualification_authorization,
        drive_authorization=drive_authorization,
        destination_confirmation=destination_confirmation,
        mountpoint=support.COLAB_DRIVE_MOUNTPOINT,
        drive_root=support.COLAB_DRIVE_ROOT,
        local_staging_root=Path("/content/res142-drive-qualification-local"),
        drive_mount=colab_drive.mount,
        drive_is_mounted=support.is_colab_drive_mounted,
        drive_flush_and_unmount=colab_drive.flush_and_unmount,
        verify_in_drive_ui=verify_in_independent_drive_ui,
    )
except Exception as exc:
    print(json.dumps({"status": "incomplete", "error": support.sanitize_output(exc, limit=1000)}, indent=2))
    raise

print(json.dumps(qualification_result, indent=2, sort_keys=True))
if qualification_result["status"] != "complete":
    raise RuntimeError("CPU-only Drive qualification is incomplete; do not proceed or claim qualification.")